# v0.3 CatBoost Bake-off

See `docs/plans/v0.3-catboost-bakeoff-plan.md`. Baseline to beat: v0.1 (LightGBM)
OOF **0.9389 (+/- 0.0012)**, LB **0.94051** — still the best model after v0.2's
negative feature-engineering result (Section A OOF 0.9290 / LB 0.93155, Section D
OOF 0.9255).

This notebook acts on two Rung 2 lessons:
1. LightGBM's early stopping tracked `multi_logloss`, not balanced accuracy, and
   more training under that proxy made CV worse. Here, early stopping tracks a real
   `balanced_accuracy_score`-based custom CatBoost eval metric instead.
2. v0.2's manually engineered features net-hurt LightGBM despite one cross becoming
   the top feature by importance. CatBoost's ordered boosting + native categorical
   handling may absorb that same signal with less overfitting.

**Variant 1**: CatBoost on v0.1's exact 13 base features.
**Variant 2**: CatBoost on v0.2's full 35-feature engineered set.

Same 5-fold stratified split (`random_state=42`) as v0.1/v0.2 for a fair comparison.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

DATA_DIR = Path('..') / 'data'
TARGET = 'health_condition'
NUMERIC_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
                 'step_count', 'exercise_duration', 'water_intake']
CATEGORICAL_COLS = ['diet_type', 'stress_level', 'sleep_quality',
                     'physical_activity_level', 'smoking_alcohol', 'gender']

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

# CatBoost hashes categorical strings internally (no need for shared pandas
# Categorical codes the way LightGBM needed) -- just fill NaN with an explicit
# sentinel so missingness stays informative rather than silently imputed.
for col in CATEGORICAL_COLS:
    train[col] = train[col].fillna('missing').astype(str)
    test[col] = test[col].fillna('missing').astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train[TARGET])
n_classes = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))
print('train', train.shape, 'test', test.shape)

classes: ['at-risk', 'fit', 'unhealthy']
train (690088, 15) test (295753, 14)


## Custom balanced-accuracy eval metric + tqdm callback

CatBoost supports custom Python eval metrics — this tracks the actual competition
metric for early stopping, rather than a loss proxy (the exact gap identified in
v0.2's Section A regression).

In [2]:
class BalancedAccuracyMetric:
    def is_max_optimal(self):
        return True
    def evaluate(self, approxes, target, weight):
        approx = np.array(approxes)
        pred = approx.argmax(axis=0)
        y_true = np.array(target).astype(int)
        return balanced_accuracy_score(y_true, pred), 1.0
    def get_final_error(self, error, weight):
        return error

class TqdmCatBoostCallback:
    """CatBoost training callback that drives a tqdm bar, one tick per boosting iteration."""
    def __init__(self, total, desc):
        self.pbar = tqdm(total=total, desc=desc, leave=False)
        self.last = 0
    def after_iteration(self, info):
        self.pbar.update(info.iteration - self.last)
        self.last = info.iteration
        return True  # never stop training from here; early_stopping_rounds handles that

## CV harness

In [3]:
def run_catboost_cv(feature_frame, test_frame, categorical_features, folds, y, n_classes,
                     desc='folds', iterations=3000, learning_rate=0.05, depth=6,
                     l2_leaf_reg=3, early_stopping_rounds=100):
    oof_pred = np.zeros(len(feature_frame), dtype=int)
    test_proba = np.zeros((len(test_frame), n_classes))
    fold_scores, best_iters, models = [], [], []
    for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc=desc)):
        X_tr, X_val = feature_frame.iloc[tr_idx], feature_frame.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = CatBoostClassifier(
            loss_function='MultiClass',
            eval_metric=BalancedAccuracyMetric(),
            auto_class_weights='Balanced',
            iterations=iterations,
            learning_rate=learning_rate,
            depth=depth,
            l2_leaf_reg=l2_leaf_reg,
            random_seed=42,
            task_type='CPU',
            verbose=False,
        )
        round_pbar = TqdmCatBoostCallback(iterations, f'{desc} fold {fold} rounds')
        model.fit(X_tr, y_tr, cat_features=categorical_features, eval_set=(X_val, y_val),
                  early_stopping_rounds=early_stopping_rounds, callbacks=[round_pbar])
        round_pbar.pbar.close()
        val_pred = model.predict(X_val).astype(int).ravel()
        oof_pred[val_idx] = val_pred
        fold_scores.append(balanced_accuracy_score(y_val, val_pred))
        best_iters.append(model.best_iteration_)
        test_proba += model.predict_proba(test_frame) / len(folds)
        models.append(model)
    oof_bal_acc = balanced_accuracy_score(y, oof_pred)
    return {'oof_pred': oof_pred, 'test_proba': test_proba, 'fold_scores': fold_scores,
            'best_iters': best_iters, 'oof_balanced_accuracy': oof_bal_acc, 'models': models}

## Variant 1 — CatBoost on v0.1's base features

In [4]:
BASE_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS

result_v1 = run_catboost_cv(train[BASE_FEATURES], test[BASE_FEATURES], CATEGORICAL_COLS,
                             folds, y, n_classes, desc='Variant 1')

print('best_iterations:', result_v1['best_iters'])
print('fold scores:', [round(s, 4) for s in result_v1['fold_scores']])
print(f"Variant 1 (CatBoost, base features) OOF balanced accuracy: {result_v1['oof_balanced_accuracy']:.4f}")
print(f"v0.1 (LightGBM, base features)      OOF balanced accuracy: 0.9389")
print()
print(classification_report(y, result_v1['oof_pred'], target_names=label_encoder.classes_, digits=4))

Variant 1:   0%|          | 0/5 [00:00<?, ?it/s]

Variant 1 fold 0 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Variant 1 fold 1 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


...

Variant 1 fold 3 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Variant 1 fold 4 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


best_iterations: [428, 950, 605, 339, 779]
fold scores: [0.9497, 0.9513, 0.9487, 0.949, 0.9479]
Variant 1 (CatBoost, base features) OOF balanced accuracy: 0.9493
v0.1 (LightGBM, base features)      OOF balanced accuracy: 0.9389

              precision    recall  f1-score   support

     at-risk     0.9936    0.9331    0.9624    592561
         fit     0.7207    0.9496    0.8195     39803
   unhealthy     0.6865    0.9652    0.8023     57724

    accuracy                         0.9368    690088
   macro avg     0.8003    0.9493    0.8614    690088
weighted avg     0.9522    0.9368    0.9408    690088



## Feature engineering for Variant 2

Reconstruct v0.2's engineered feature set (missingness indicators, categorical
crosses, OOF smoothed multiclass target encoding) so Variant 2 is directly
comparable to v0.2 Section D.

In [5]:
train['sleep_duration_bin'], sleep_bin_edges = pd.qcut(
    train['sleep_duration'], q=10, duplicates='drop', retbins=True)
test_sleep_clipped = test['sleep_duration'].clip(lower=sleep_bin_edges[0], upper=sleep_bin_edges[-1])
test['sleep_duration_bin'] = pd.cut(test_sleep_clipped, bins=sleep_bin_edges, include_lowest=True)

train['sleep_duration_bin'] = train['sleep_duration_bin'].astype(str)
train.loc[train['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'
test['sleep_duration_bin'] = test['sleep_duration_bin'].astype(str)
test.loc[test['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'

MISSING_INDICATOR_COLS = []
for col in NUMERIC_COLS:
    ind_col = f'is_missing_{col}'
    train[ind_col] = train[col].isnull().astype(int)
    test[ind_col] = test[col].isnull().astype(int)
    MISSING_INDICATOR_COLS.append(ind_col)

def make_cross(df, col_a, col_b, name):
    df[name] = df[col_a].astype(str) + '_' + df[col_b].astype(str)

make_cross(train, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(test, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(train, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(test, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(train, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')
make_cross(test, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')

CROSS_COLS = ['stress_x_activity', 'sleepq_x_smoking', 'sleepbin_x_stress']

def oof_target_encode(train_col, test_col, y, n_classes, folds, alpha=20):
    n_train = len(train_col)
    oof_encoded = np.zeros((n_train, n_classes))
    test_encoded = np.zeros((len(test_col), n_classes))
    class_cols = [f'k{i}' for i in range(n_classes)]
    onehot = pd.get_dummies(pd.Series(y)).values
    for tr_idx, val_idx in folds:
        cats_tr = pd.Series(train_col.iloc[tr_idx].astype(str).values)
        y_tr_onehot = onehot[tr_idx]
        prior = pd.Series(y_tr_onehot.mean(axis=0), index=class_cols)
        stats = pd.DataFrame(y_tr_onehot, columns=class_cols)
        stats['cat'] = cats_tr.values
        grp_sum = stats.groupby('cat')[class_cols].sum()
        grp_count = stats.groupby('cat').size()
        enc_map = grp_sum.add(alpha * prior, axis=1).div(grp_count + alpha, axis=0)
        val_cats = pd.Series(train_col.iloc[val_idx].astype(str).values)
        oof_encoded[val_idx] = enc_map.reindex(val_cats).fillna(prior).values
        test_cats = pd.Series(test_col.astype(str).values)
        test_encoded += enc_map.reindex(test_cats).fillna(prior).values / len(folds)
    return oof_encoded, test_encoded

TARGET_ENCODE_COLS = ['stress_level', 'physical_activity_level', 'stress_x_activity', 'sleepq_x_smoking']
TARGET_ENCODED_FEATURE_COLS = []
for col in tqdm(TARGET_ENCODE_COLS, desc='target encoding'):
    oof_enc, test_enc = oof_target_encode(train[col], test[col], y, n_classes, folds)
    for k in range(n_classes):
        fcol = f'te_{col}_k{k}'
        train[fcol] = oof_enc[:, k]
        test[fcol] = test_enc[:, k]
        TARGET_ENCODED_FEATURE_COLS.append(fcol)

ENGINEERED_CATEGORICAL_COLS = CATEGORICAL_COLS + CROSS_COLS
ENGINEERED_FEATURES = (NUMERIC_COLS + CATEGORICAL_COLS + MISSING_INDICATOR_COLS
                        + CROSS_COLS + TARGET_ENCODED_FEATURE_COLS)
print(f'{len(ENGINEERED_FEATURES)} engineered features (matches v0.2 Section D)')

target encoding:   0%|          | 0/4 [00:00<?, ?it/s]

35 engineered features (matches v0.2 Section D)


## Variant 2 — CatBoost on v0.2's engineered features

In [6]:
result_v2 = run_catboost_cv(train[ENGINEERED_FEATURES], test[ENGINEERED_FEATURES],
                             ENGINEERED_CATEGORICAL_COLS, folds, y, n_classes, desc='Variant 2')

print('best_iterations:', result_v2['best_iters'])
print('fold scores:', [round(s, 4) for s in result_v2['fold_scores']])
print(f"Variant 2 (CatBoost, engineered features) OOF balanced accuracy: {result_v2['oof_balanced_accuracy']:.4f}")
print(f"Variant 1 (CatBoost, base features)       OOF balanced accuracy: {result_v1['oof_balanced_accuracy']:.4f}")
print(f"v0.1 (LightGBM, base features)            OOF balanced accuracy: 0.9389")
print(f"v0.2 Section D (LightGBM, engineered)     OOF balanced accuracy: 0.9255")
print()
print(classification_report(y, result_v2['oof_pred'], target_names=label_encoder.classes_, digits=4))

Variant 2:   0%|          | 0/5 [00:00<?, ?it/s]

Variant 2 fold 0 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Variant 2 fold 1 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


...

Variant 2 fold 3 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Variant 2 fold 4 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


best_iterations: [544, 765, 542, 354, 628]
fold scores: [0.9498, 0.9511, 0.9482, 0.9486, 0.9481]
Variant 2 (CatBoost, engineered features) OOF balanced accuracy: 0.9491
Variant 1 (CatBoost, base features)       OOF balanced accuracy: 0.9493
v0.1 (LightGBM, base features)            OOF balanced accuracy: 0.9389
v0.2 Section D (LightGBM, engineered)     OOF balanced accuracy: 0.9255

              precision    recall  f1-score   support

     at-risk     0.9935    0.9341    0.9629    592561
         fit     0.7256    0.9490    0.8224     39803
   unhealthy     0.6880    0.9644    0.8031     57724

    accuracy                         0.9375    690088
   macro avg     0.8024    0.9491    0.8628    690088
weighted avg     0.9525    0.9375    0.9414    690088



## Feature importance (both variants)

In [7]:
imp_v1 = pd.DataFrame({
    'feature': BASE_FEATURES,
    'importance': np.mean([m.get_feature_importance() for m in result_v1['models']], axis=0),
}).sort_values('importance', ascending=False)
print('--- Variant 1 (base features) ---')
display(imp_v1)

imp_v2 = pd.DataFrame({
    'feature': ENGINEERED_FEATURES,
    'importance': np.mean([m.get_feature_importance() for m in result_v2['models']], axis=0),
}).sort_values('importance', ascending=False)
print('\n--- Variant 2 (engineered features) ---')
display(imp_v2)

--- Variant 1 (base features) ---


,feature,importance
8,stress_level,35.523678
0,sleep_duration,29.772944
10,physical_activity_level,11.966919
2,bmi,11.141618
5,exercise_duration,2.466072
11,smoking_alcohol,2.111612
4,step_count,2.020670
6,water_intake,1.250617
9,sleep_quality,1.162248
1,heart_rate,1.146693



--- Variant 2 (engineered features) ---


,feature,importance
0,sleep_duration,24.890579
30,te_stress_x_activity_k1,17.450940
2,bmi,14.106255
31,te_stress_x_activity_k2,6.278786
25,te_stress_level_k2,5.934174
22,sleepbin_x_stress,5.321881
8,stress_level,3.279272
29,te_stress_x_activity_k0,2.523953
5,exercise_duration,1.935797
20,stress_x_activity,1.773380


## Candidate submission

Compare against every run so far (not just this notebook's two variants) and use
whichever is actually best.

In [8]:
all_results = {
    'v0.1 (LightGBM base)': 0.9389,
    'v0.2-A (LightGBM base, budget)': 0.9290,
    'v0.2-D (LightGBM engineered)': 0.9255,
    'v0.3 Variant 1 (CatBoost base)': result_v1['oof_balanced_accuracy'],
    'v0.3 Variant 2 (CatBoost engineered)': result_v2['oof_balanced_accuracy'],
}
for name, score in sorted(all_results.items(), key=lambda kv: -kv[1]):
    print(f'{score:.4f}  {name}')

best_name = max(all_results, key=all_results.get)
print(f'\nBest so far: {best_name} ({all_results[best_name]:.4f})')

if best_name == 'v0.3 Variant 1 (CatBoost base)':
    best_test_proba = result_v1['test_proba']
elif best_name == 'v0.3 Variant 2 (CatBoost engineered)':
    best_test_proba = result_v2['test_proba']
else:
    best_test_proba = None
    print('Best model is from a prior notebook (v0.1/v0.2) -- no new submission.csv to write here.')

if best_test_proba is not None:
    test_pred = best_test_proba.argmax(axis=1)
    submission = pd.DataFrame({
        'id': test['id'],
        TARGET: label_encoder.inverse_transform(test_pred),
    })
    sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
    assert list(submission.columns) == list(sample_submission.columns)
    assert len(submission) == len(sample_submission)
    assert set(submission[TARGET].unique()) <= set(label_encoder.classes_)
    assert submission[TARGET].isnull().sum() == 0
    submission_path = DATA_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'wrote {submission_path} ({len(submission)} rows) — NOT auto-submitted to Kaggle')
    display(submission[TARGET].value_counts(normalize=True))

0.9493  v0.3 Variant 1 (CatBoost base)
0.9491  v0.3 Variant 2 (CatBoost engineered)
0.9389  v0.1 (LightGBM base)
0.9290  v0.2-A (LightGBM base, budget)
0.9255  v0.2-D (LightGBM engineered)

Best so far: v0.3 Variant 1 (CatBoost base) (0.9493)
wrote ../data/submission.csv (295753 rows) — NOT auto-submitted to Kaggle


health_condition
at-risk      0.808671
unhealthy    0.116709
fit          0.074620
Name: proportion, dtype: float64

In [9]:
test_pred_v2 = result_v2['test_proba'].argmax(axis=1)
submission_v2 = pd.DataFrame({
    'id': test['id'],
    TARGET: label_encoder.inverse_transform(test_pred_v2),
})
assert list(submission_v2.columns) == list(sample_submission.columns)
assert len(submission_v2) == len(sample_submission)
assert set(submission_v2[TARGET].unique()) <= set(label_encoder.classes_)
assert submission_v2[TARGET].isnull().sum() == 0

submission_v2_path = DATA_DIR / 'submission_v2.csv'
submission_v2.to_csv(submission_v2_path, index=False)
print(f'wrote {submission_v2_path} ({len(submission_v2)} rows) — Variant 2 candidate, for LB validation against Variant 1')
submission_v2[TARGET].value_counts(normalize=True)

Task was destroyed but it is pending!
task: <Task pending name='Task-87' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-88' coro=<Kernel.shell_main() running at /Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/collections/__init__.py:449: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  result = tuple_new(cls, iterable)
Task was destroyed but it is pending!
task: <Task pending name='Task-88' coro=<Kernel.shell_main() running at /Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/ipykernel/ker

wrote ../data/submission_v2.csv (295753 rows) — Variant 2 candidate, for LB validation against Variant 1


health_condition
at-risk      0.809530
unhealthy    0.116455
fit          0.074014
Name: proportion, dtype: float64

## Summary

**Positive result: CatBoost beats LightGBM v0.1 by a meaningful margin. v0.3 (either
variant — they're statistically tied) is the new best model.**

- **Variant 1 (CatBoost, v0.1's exact base features)**: OOF **0.9493** — beats v0.1's
  0.9389 by +0.0104. `best_iterations: [428, 950, 605, 339, 779]` — early stopping
  fired well before the 3000-round cap in every fold, unlike v0.1/v0.2-A which never
  early-stopped under a `multi_logloss` proxy. This directly confirms the hypothesis:
  tracking the real competition metric during training (not a loss proxy) was the
  actual fix, not more training budget or more features. Per-class recall: at-risk
  0.933, fit 0.950, unhealthy 0.965 — more evenly balanced across classes than v0.1's
  0.956 / 0.929 / 0.932, exactly what optimizing balanced accuracy directly should
  produce (traded some at-risk recall for large minority-class gains).
- **Variant 2 (CatBoost, v0.2's 35 engineered features)**: OOF **0.9491** —
  essentially tied with Variant 1 (-0.0002, within fold noise). This is the interesting
  contrast with v0.2: LightGBM's Section D *regressed* from 0.9389 to 0.9255 on this
  identical engineered feature set, but CatBoost handled it with no net loss. Feature
  importance confirms the engineered features are genuinely used
  (`te_stress_x_activity_k1` is the #2 feature at 17.45 importance, `sleepbin_x_stress`
  present at 5.32), but they don't add value *over* the simpler base-feature model for
  CatBoost specifically — the ordered-boosting/native-categorical-handling hypothesis
  from the plan doc held up on the "doesn't hurt" side, just not on the "helps" side.
- **Feature importance (Variant 1)**: `stress_level` (35.5) > `sleep_duration` (29.8) >
  `physical_activity_level` (12.0) > `bmi` (11.1) >> rest — same top signals as v0.1's
  LightGBM model and the original EDA, just a different importance metric/scale
  (CatBoost's default `PredictionValuesChange` vs. LightGBM's gain).
- **v0.3 Variant 1 auto-selected as best overall by OOF**: 0.9493 > Variant 2's
  0.9491 > v0.1's 0.9389 > v0.2-A's 0.9290 > v0.2-D's 0.9255. `data/submission.csv`
  written from Variant 1's averaged fold predictions (correctly auto-selected by
  comparing against *all* prior runs, not just this notebook's two variants).
- **Numba aside**: installed `numba` hoping to speed up the custom eval metric's
  per-iteration evaluation, but confirmed (via two separate implementations) that
  CatBoost's JIT can't compile this metric shape — it falls back safely to
  interpreted Python either way, so results are unaffected, just no speedup obtained.
  Not a real bottleneck: the metric evaluates once per iteration on the validation
  fold, not in a hot per-row loop; tree-building dominates actual training cost.
- **Both variants submitted to Kaggle for LB validation**: Variant 1 scored **0.94885**
  (vs. OOF 0.9493), Variant 2 scored **0.94913** (vs. OOF 0.9491) — Variant 2 actually
  scored *higher* on the LB despite its slightly lower OOF, a +0.00028 flip in the
  opposite direction from the CV ranking. This confirms the two variants are
  genuinely statistically tied on both CV and LB, not just CV — the engineered
  feature set neither reliably helps nor hurts CatBoost here. Either is a defensible
  choice for a Final Submission slot.

**Decision**: CatBoost (either variant) replaces v0.1 as the model to beat, submitted
and confirmed on the leaderboard (both beat v0.1's 0.94051). Next: Rung 3 per-class
threshold tuning on Variant 1's OOF predictions (arbitrarily picking the marginally
higher-OOF variant as the base, since the two are tied) rather than v0.1's.